# Full Karpathy prompt cache trước XE từ đầu
**Accelerator None, Internet ON.** CPU map liên kết và frozenCLIP chỉ cho text mới; không UPT inference, caption training, TEST, API Secret hoặc checkpointload.

Gắn4Datasets: `csa-full-roi-karpathy-v134`, `upt-prompts-full-all-actions-v2`, `objectdetectionecache`, `csa-prompt-pilot-inputs-v169`. Sửa4path ở cell đầu nếu mount khác.
Kiểm toàn bộ113287TRAIN+5000VALID, cùngIoU/instance/cap8object/3relation. Giữ318vector169 byte-identical; nếu cótext mới chỉ encode phần mới cùngCLIP/revision/assetSHA/frozenFP32CPU. Không lấy prompt từ captionthamchiếu.
Fullmapping phải giữ nguyên các trường ngữ nghĩa và float32input của15000record169. Sai số FP64 chỉ ở IoU/roi_bbox/geometry ≤1e-12 và castfloat32 giốnghệt mới được nhận; diagnostic ghi rõ trường lệch. ID/text/boxnguồn/S/O/ROI/confidence/threshold luônso chínhxác. Dừng nếu thiếudữliệu hoặc đổi semantic/input; không coi ảnhthiếuprompt lànegative/no_objects.
Chạy Save Version → Save & Run All vớiCPU. Cap4h là giới hạn dừng, không phảiETA/Kagglequota. Giữwholeoutput, gửireviewZIP. Sau review sẽ lưuDataset trước hai nhánhXEfromscratch; notebook này chưa huấn luyện và không thay kết quả172/56.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
FULL_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-roi-karpathy-v134')
PROMPTS=Path('/kaggle/input/datasets/ducanh2403/upt-prompts-full-all-actions-v2/prompt_cache.json')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
BASE_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-pilot-inputs-v169')
OUTPUT=Path('/kaggle/working/csa_prompt_full_karpathy_cache_v1')
WALL_HOURS=4 # CPU STOPcap, not ETA or a claim about Kaggle session limits.
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600
for p in (FULL_ROOT,PROMPTS,DETECTIONS,BASE_ROOT):
    if not p.exists():raise FileNotFoundError('Attach all4inputDatasets before starting: '+str(p))
if OUTPUT.exists():raise RuntimeError('Preserve existing output; fresh runtime/output path required')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<6*1024**3:raise RuntimeError('Need6GiB free for readonly metadata/download/copies; no files removed')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='csa-full-prompt-',dir='/tmp'));LOG=OUTPUT/'notebook.log'
REVISION='f1d55c06dabfe354b7214ed37873a49fd8c20326'
MAPPING={'experiments/csa_paper/prompt_adapter_data.py': 'prompt_adapter_data.py', 'experiments/csa_paper/prompt_adapter_model.py': 'prompt_adapter_model.py', 'experiments/csa_paper/run_prompt_mapping_audit.py': 'run_prompt_mapping_audit.py', 'experiments/csa_paper/test_prompt_adapter_data.py': 'test_prompt_adapter_data.py', 'experiments/csa_paper/test_prompt_adapter_model.py': 'test_prompt_adapter_model.py', 'experiments/csa_paper/make_prompt_adapter_notebook.py': 'make_prompt_adapter_notebook.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/audit_assets.py': 'audit_assets.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/model.py': 'model.py', 'experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json': 'SAVE_FULL_ROI_V134_PACKAGE.json', 'experiments/csa_paper/prompt_text_cache.py': 'prompt_text_cache.py', 'experiments/csa_paper/test_prompt_text_cache_data.py': 'test_prompt_text_cache_data.py', 'experiments/csa_paper/make_prompt_text_cache_notebook.py': 'make_prompt_text_cache_notebook.py', 'experiments/csa_paper/make_smoke_notebook.py': 'make_smoke_notebook.py', 'experiments/csa_paper/RERANK_TEST_INPUTS.json': 'RERANK_TEST_INPUTS.json', 'experiments/csa_paper/RECOVER_PROMPT_TEXT_167_CELL.py': 'RECOVER_PROMPT_TEXT_167_CELL.py', 'experiments/csa_paper/full_prompt_cache_data.py': 'full_prompt_cache_data.py', 'experiments/csa_paper/run_full_prompt_cache.py': 'run_full_prompt_cache.py', 'experiments/csa_paper/test_full_prompt_cache_data.py': 'test_full_prompt_cache_data.py', 'experiments/csa_paper/make_full_prompt_cache_notebook.py': 'make_full_prompt_cache_notebook.py', 'experiments/csa_paper/PROMPT_SCRATCH_XE_PLAN.md': 'PROMPT_SCRATCH_XE_PLAN.md', 'experiments/csa_paper/recover_full_prompt_mapping.py': 'recover_full_prompt_mapping.py', 'captioning/__init__.py': 'captioning/__init__.py', 'captioning/upt_prompts.py': 'captioning/upt_prompts.py', 'captioning/upt_action_text.py': 'captioning/upt_action_text.py', 'captioning/selected_relation_prompts.py': 'captioning/selected_relation_prompts.py', 'captioning/supervised_relations.py': 'captioning/supervised_relations.py'}
HASHES={'experiments/csa_paper/prompt_adapter_data.py': 'd8b8b61a518d9d4e2d413386d08e60963586bf3b17a16e5c96ee3f54eeaf36c0', 'experiments/csa_paper/prompt_adapter_model.py': '83e5fdb2662940ded06e56df3a0fdf512dc1046b2352e563b59f22a6117e245c', 'experiments/csa_paper/run_prompt_mapping_audit.py': 'ea35c2a582e8903d692f6aaa19735075b474fe108ea4c6ddac467a25b545af5e', 'experiments/csa_paper/test_prompt_adapter_data.py': 'a774dab306aec09aadea869953b1cc04e5592edf1e365b53f02e73258be92851', 'experiments/csa_paper/test_prompt_adapter_model.py': '152d7419238d9484c5b16e84fd987f3e9f68a57649f17019e5532860e144554a', 'experiments/csa_paper/make_prompt_adapter_notebook.py': 'b553a7d68efda2c679b2b0b470fa63f11d772782ce112d186d10b4c05ff55b34', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/audit_assets.py': '5be2fdf1ee094ef8b23a547b16f423ed3603f7c80b2fe5f2d46cb9224cb90959', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json': 'a73636ca75806d6b1d7b4d88f80f3ce1c59b3061eb3e9a5e730097ddd89c505c', 'experiments/csa_paper/prompt_text_cache.py': '0bd52c07b44d201a631e6383af111c21d7b3ef2e2ed676f5bc13fc53306e0f23', 'experiments/csa_paper/test_prompt_text_cache_data.py': 'e5cfe2b6ea75c29661265b195d5d7377eb0c5d3cbe497807ed3300a95c5a89d5', 'experiments/csa_paper/make_prompt_text_cache_notebook.py': 'f81983cd170632bdd5b787d3ffd1168be9d3018562d072bd9fc28a6450d84e82', 'experiments/csa_paper/make_smoke_notebook.py': '9de13e286efb3ddd29a2eed331219a9e60dcb384fc2f28e062f64dd8dd1c6054', 'experiments/csa_paper/RERANK_TEST_INPUTS.json': '33395ec35a0803ebf711737f0bdba3b6be8c16244f4c292057658e0fa3feaf46', 'experiments/csa_paper/RECOVER_PROMPT_TEXT_167_CELL.py': 'f91972596641c314ff3c7817889e63429cbb3fcc0f20950fb5db101311d8463a', 'experiments/csa_paper/full_prompt_cache_data.py': '7ecf20920c39bac3caa92a14b01723ae27308a1858e94bd0e19fbafcd399f33b', 'experiments/csa_paper/run_full_prompt_cache.py': '360c7342b26715c877e344f418d837237e526af3a4446bb4bc6632c071d099a5', 'experiments/csa_paper/test_full_prompt_cache_data.py': 'a8ca1d0264873696d915aac7688560a859e8ef4167aff628373587daa0b1e297', 'experiments/csa_paper/make_full_prompt_cache_notebook.py': '7ac4d604ed1fe6583a3c3bfb614da8ba5b61658e3ce9c552d95d5415253ef765', 'experiments/csa_paper/PROMPT_SCRATCH_XE_PLAN.md': 'a43687e9edca06625f0a75f29a0a9c50df2793a4795d43f321168e6dfe28dcb8', 'experiments/csa_paper/recover_full_prompt_mapping.py': 'b034be6fc1a5972b1ec690c00dcb85b39d58d0357a3d8fd7c8a199659091f6c3', 'captioning/__init__.py': 'ad259f50a32265d298d027a888f801d378c4669313b8dab06e0653978ce30633', 'captioning/upt_prompts.py': '25bb24346d2b2c649ee7091c72d5def060ddf2c290524a35ef3d301d274fb65c', 'captioning/upt_action_text.py': 'ca0d8f1dc75be3ccfd589fcddc962e87a43222751a4d1beb68098cfa213d526c', 'captioning/selected_relation_prompts.py': 'de13dfbd80e4d4b8c08cd3e40407d5134cb1a5108c58f3eebfd4b74e237179d2', 'captioning/supervised_relations.py': '3e29fd60bbf166ae078581fb9ad34da6ce4462ee287f1c4eb9bbda2abd91f984'}

for original,destination in MAPPING.items():
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('CPU parent cap during source download')
    raw=urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+original,timeout=min(90,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=HASHES[original]:raise ValueError('Changed pinned source:'+original)
    target=SOURCE/destination;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2),encoding='utf-8')
os.environ['PYTHONPATH']=str(SOURCE);os.environ['OMP_NUM_THREADS']='1'
CONFIG=SOURCE/'config.json';MAPPING_ROOT=OUTPUT/'mapping'
cfg=dict(full_root=str(FULL_ROOT),prompts=str(PROMPTS),detections=str(DETECTIONS),base_root=str(BASE_ROOT),
    output=str(OUTPUT),mapping_root=str(MAPPING_ROOT),hf_cache=str(SOURCE/'hf-cache'))
CONFIG.write_text(json.dumps(cfg),encoding='utf-8')
MAP_CONFIG=SOURCE/'mapping_config.json'
MAP_CONFIG.write_text(json.dumps(dict(full_root=str(FULL_ROOT),prompts=str(PROMPTS),detections=str(DETECTIONS),
    output=str(MAPPING_ROOT),scope='full',wall_seconds=10800)),encoding='utf-8')


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('CPU parent cap exceeded')
    timeout=min(timeout,max(.01,PARENT_DEADLINE-time.monotonic()))
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Full prompt CPU active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Full prompt CPU phase/parent cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
try:
    run([sys.executable,'-u',SOURCE/'test_prompt_adapter_data.py'],timeout=120)
    run([sys.executable,'-u',SOURCE/'test_prompt_text_cache_data.py'],timeout=120)
    run([sys.executable,'-u',SOURCE/'test_full_prompt_cache_data.py'],timeout=120)
    # Verify reusable169 bytes and embedding space before processing full Karpathy.
    run([sys.executable,'-c',"from full_prompt_cache_data import checked_base;from pathlib import Path;import sys;r,t,a,m=checked_base(Path(sys.argv[1]));print('Exact169 base verified:',len(t),a.shape,'No weights/model loaded.',flush=True)",BASE_ROOT],timeout=180)
    run([sys.executable,'-u',SOURCE/'run_prompt_mapping_audit.py','--config',MAP_CONFIG],timeout=10800)
    run([sys.executable,'-u',SOURCE/'run_full_prompt_cache.py','inspect','--config',CONFIG],timeout=900)
    plan=json.loads((OUTPUT/'cache_plan.json').read_text(encoding='utf-8'))
    print('NEW text vectors to encode:',plan['encode_new_count'],'; reuse318 unchanged.',flush=True)
    if plan['encode_new_count']:
        DEPS=SOURCE/'hf-deps'
        run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,
            'transformers==4.57.1','tokenizers==0.22.1','huggingface-hub==0.36.2','safetensors==0.6.2','ftfy==6.3.1'],timeout=300)
        os.environ['PYTHONPATH']=str(DEPS)+os.pathsep+str(SOURCE)
        run([sys.executable,'-c',"from importlib.metadata import version;expected={'transformers':'4.57.1','tokenizers':'0.22.1','huggingface-hub':'0.36.2','safetensors':'0.6.2'};assert all(version(k)==v for k,v in expected.items());print('Isolated HF dependency versions PASS')"],timeout=60)
    run([sys.executable,'-u',SOURCE/'run_full_prompt_cache.py','cache','--config',CONFIG],timeout=3600)
    receipt=json.loads((OUTPUT/'cache_receipt.json').read_text(encoding='utf-8'))
    complete=receipt['status']=='FULL_PROMPT_TEXT_CACHE_COMPLETE'
    if not complete:raise RuntimeError('Incomplete full cache')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,training=False,test_used=False,caption_weights_loaded=False)),encoding='utf-8')
    REVIEW=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        for name in ('cache_plan.json','cache_manifest.json','cache_receipt.json','mapping_receipt.json','reuse_report.json',
            'summary.json','cohort.json','review_examples.json','source_receipt.json','failure.txt','notebook.log','run_status.json',
            'comparison_diagnostic.json','recovery_receipt.json','mapping_source_receipt.json'):
            if (OUTPUT/name).is_file():z.write(OUTPUT/name,name)
        # Preserve mapping diagnostics even when the later cache phase did not start.
        for name in ('summary.json','receipt.json','cohort.json','review_examples.json','failure.json'):
            if (MAPPING_ROOT/name).is_file():z.write(MAPPING_ROOT/name,'mapping/'+name)
    print('SEND REVIEW:',REVIEW,'\nKEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    print('FULL prompt cache ready for review; no caption training.' if complete else 'INCOMPLETE; preserve output/logs; do not use a partial cache.',flush=True)
